# Low-wind turbines and system resilience — 2013, 24h

**Two systems** (design runs, perfect foresight):
- `normal_wt`: normal wind turbines only
- `with_lwt`: normal + low-wind turbines (cost factor 0.7)

**Each system is then operated in several ways** (capacities frozen):
- *Perfect foresight*: the design run itself
- *Rolling horizon, 2 weeks*: `_op.nc`
- *Rolling horizon, 2 weeks + 1 week overlap*: `_op_overlap.nc`
- *(optional) Rolling horizon, 3 weeks*: `_op_21d.nc`, if you saved it

Shortfalls appear as **load shedding** (carrier `load`), i.e. unserved energy.

**Comparisons in this notebook**
1. Design: `normal_wt` vs `with_lwt` (what is built, cost, generation)
2. Check: is the design itself reliable?
3. Resilience: unserved energy for every system × operation mode
4. When and where shortages happen (main operation mode)
5. Why: hydrogen storage with perfect vs limited foresight

In [ ]:
import os
import pypsa
import pandas as pd
import matplotlib.pyplot as plt

RUN = "results/lwt_resilience"
NAME = "base_s_40__Co2L0-24h-lv1.25-wcf0.7_2050"
SCENARIOS = ["normal_wt", "with_lwt"]

# Operation modes: label -> file suffix ("" = design run = perfect foresight)
MODES = {
    "Perfect foresight": "",
    "Rolling 2w": "_op",
    "Rolling 2w + 1w overlap": "_op_overlap",
    "Rolling 3w": "_op_21d",
}
MAIN_MODE = "Rolling 2w + 1w overlap"  # used for the detailed plots in sections 4-5

nets = {}
for mode, suffix in MODES.items():
    for s in SCENARIOS:
        f = f"{RUN}/{s}/networks/{NAME}{suffix}.nc"
        if os.path.exists(f):
            nets[(mode, s)] = pypsa.Network(f)
        else:
            print(f"(missing, skipped) {f}")

design = {s: nets[("Perfect foresight", s)] for s in SCENARIOS}
if (MAIN_MODE, SCENARIOS[0]) not in nets:
    MAIN_MODE = "Rolling 2w"
    print("Overlap runs not found, using 'Rolling 2w' for detailed plots.")
op = {s: nets[(MAIN_MODE, s)] for s in SCENARIOS}

print("\nLoaded:")
for (mode, s), n in nets.items():
    print(f"  {mode:26s} {s:10s} snapshots: {len(n.snapshots)}")

In [ ]:
def shedding(n):
    """Load shedding per snapshot [MW] for every shedding generator (carrier 'load')."""
    gens = n.generators.index[n.generators.carrier == "load"]
    return n.generators_t.p[gens]

def unserved_summary(n):
    shed = shedding(n)
    w = n.snapshot_weightings.generators
    demand = n.loads_t.p.multiply(w, axis=0).sum().sum()
    unserved = shed.multiply(w, axis=0).sum().sum()
    return {
        "Unserved energy [TWh]": unserved / 1e6,
        "Share of demand [%]": unserved / demand * 100,
        "Hours with shortage [h]": w[shed.sum(axis=1) > 1e-3].sum(),
        "Peak shortage [GW]": shed.sum(axis=1).max() / 1e3,
    }

## 1. Design: what gets built, normal_wt vs with_lwt

In [ ]:
def capacities(n):
    gen = n.generators.groupby("carrier").p_nom_opt.sum()
    links = n.links.groupby("carrier").p_nom_opt.sum()
    stores = n.stores.groupby("carrier").e_nom_opt.sum()
    return pd.concat({"Generators [GW]": gen / 1e3,
                      "Links [GW]": links / 1e3,
                      "Stores [GWh]": stores / 1e3})

cap = pd.DataFrame({s: capacities(design[s]) for s in SCENARIOS}).fillna(0)
cap = cap.replace([float("inf"), -float("inf")], pd.NA).dropna()  # drop unbounded helpers (load shedding, CO2 atmosphere)
cap = cap[(cap > 0.1).any(axis=1)].astype(float).round(1)
cap["difference"] = cap["with_lwt"] - cap["normal_wt"]
cap

In [ ]:
wind_solar = ["onwind", "onwind_type2", "offwind-ac", "offwind-dc", "offwind-float", "solar", "solar-hsat", "solar rooftop"]
gen = pd.DataFrame({s: design[s].generators.groupby("carrier").p_nom_opt.sum() / 1e3 for s in SCENARIOS})
gen.reindex(wind_solar).fillna(0).T.plot.bar(stacked=True, figsize=(6, 4))
plt.ylabel("Installed capacity [GW]")
plt.title("Wind and solar capacity (design)")
plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()

In [ ]:
cost = pd.Series({s: design[s].objective / 1e9 for s in SCENARIOS}, name="Total system cost [bn EUR/a]")
print(cost.round(2))
print(f"Saving with low-wind turbines: {(cost['normal_wt'] - cost['with_lwt']) / cost['normal_wt'] * 100:.1f} %")

In [ ]:
def energy_by_carrier(n, carriers):
    p = n.generators_t.p.multiply(n.snapshot_weightings.generators, axis=0).sum()
    return p.groupby(n.generators.carrier).sum().reindex(carriers).fillna(0) / 1e6  # TWh

print("Annual generation [TWh] (design)")
pd.DataFrame({s: energy_by_carrier(design[s], wind_solar) for s in SCENARIOS}).round(1)

## 2. Check: is the design itself reliable?

The design (perfect foresight) should have **~0 unserved energy**. If not, load shedding was enabled in the design run and the system is not built to be fully reliable: set `solving: options: load_shedding: enable: false` and rerun the design.

In [ ]:
pd.DataFrame({s: unserved_summary(design[s]) for s in SCENARIOS}).round(3)

## 3. Resilience: unserved energy for every system and operation mode

Main comparison of the study. Read it two ways:
- **Down a column**: how much limited foresight hurts each system.
- **Across a row**: whether low-wind turbines make the system more robust.

In [ ]:
rows = {(mode, s): unserved_summary(n) for (mode, s), n in nets.items()}
table = pd.DataFrame(rows).T
table.index.names = ["Operation", "System"]
table.round(2)

In [ ]:
ue = table["Unserved energy [TWh]"].unstack("System").reindex([m for m in MODES if m in table.index.get_level_values(0)])
ue.plot.bar(figsize=(8, 4), rot=0)
plt.ylabel("Unserved energy [TWh]")
plt.title("Unserved energy by operation mode")
plt.tight_layout()

change = (ue["with_lwt"] - ue["normal_wt"]) / ue["normal_wt"].where(ue["normal_wt"] > 0) * 100
print("Change in unserved energy with low-wind turbines [%]:")
print(change.round(1))

## 4. When and where do shortages happen? (main operation mode)

In [ ]:
print("Operation mode:", MAIN_MODE)
fig, ax = plt.subplots(figsize=(10, 4))
for s in SCENARIOS:
    (shedding(op[s]).sum(axis=1) / 1e3).plot(ax=ax, label=s)
ax.set_ylabel("Unserved power [GW]")
ax.set_title(f"Shortages over the year ({MAIN_MODE})")
ax.legend()
plt.tight_layout()

In [ ]:
def shedding_by_country(n):
    shed = shedding(n).multiply(n.snapshot_weightings.generators, axis=0).sum()
    buses = n.generators.loc[shed.index, "bus"]
    country = buses.map(n.buses.country) if "country" in n.buses else buses.str[:2]
    return shed.groupby(country).sum() / 1e6  # TWh

by_country = pd.DataFrame({s: shedding_by_country(op[s]) for s in SCENARIOS}).fillna(0)
by_country = by_country[by_country.sum(axis=1) > 0]
by_country.plot.bar(figsize=(7, 4), rot=0)
plt.ylabel("Unserved energy [TWh]")
plt.title(f"Unserved energy by country ({MAIN_MODE})")
plt.tight_layout()
by_country.round(3)

## 5. Why? Hydrogen storage: perfect vs limited foresight

With perfect foresight, H2 storage is filled ahead of long low-wind periods. With limited foresight, it may be too low when they arrive.

In [ ]:
carrier = "H2"  # change to "battery" to look at batteries
fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(10, 6), sharex=True)
for ax, s in zip(axes, SCENARIOS):
    for label, n in [("Perfect foresight", design[s]), (MAIN_MODE, op[s])]:
        stores = n.stores.index[n.stores.carrier.str.startswith(carrier)]
        (n.stores_t.e[stores].sum(axis=1) / 1e6).plot(ax=ax, label=label)
    ax.set_title(s)
    ax.set_ylabel(f"{carrier} storage [TWh]")
    ax.legend()
plt.tight_layout()

## Notes

- **24h resolution** (test runs): daily averages hide short calm spells, so unserved energy is likely underestimated. Final results: 3h on Sophia.
- At 24h, each time step counts as 24 h, so "hours with shortage" is a multiple of 24 (number of days × 24).
- Foresight: 14 steps = 2 weeks; overlap run = 21 steps solved, 14 kept.
- Value of lost load: 100,000 EUR/MWh.